# ⚙️ RUPS-300 Problem Generator
### *Reasoning Under Pressure — Full Benchmark Construction*

**What this notebook does:**
- Generates 300 structured reasoning problems using `claude-sonnet-4.6`
- Validates every problem automatically with `gpt-4o`
- Saves clean `rups296_benchmark.json` ready for the main experiment

**Domain breakdown:**

| Domain | Count | Failure modes | Task type |
|--------|-------|--------------|----------|
| Financial | 100 | F1, F3, F5 | Credit risk, loan assessment, portfolio analysis |
| Clinical / ICU | 100 | F1, F2, F3, F4 | Lab interpretation, vitals, medication safety |
| HR / Supply chain | 100 | F2, F4, F5 | Attrition prediction, performance, procurement |

**Pipeline:** Generator (claude-sonnet-4.6) → Validator (gpt-4o) → rups296_benchmark.json

**Estimated runtime:** ~20–25 minutes | **API calls:** ~90 generation + ~300 validation

---

## 📦 Cell 1 — Install & imports

In [ ]:
!pip install aiohttp nest_asyncio --quiet

import requests, aiohttp, asyncio, time, json, os, re
from datetime import datetime
from collections import Counter
import nest_asyncio
nest_asyncio.apply()

print('✅ Ready')

## 🔧 Cell 2 — LLM Gateway

In [ ]:
class TokenService:
    """
    Authentication service for LLM gateway.
    
    To reproduce experiments, replace this class with direct API calls
    to your preferred LLM provider using the model names in Table 4.
    
    Supported alternatives:
      - OpenAI:    openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
      - Anthropic: anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])  
      - Groq:      groq.Groq(api_key=os.environ["GROQ_API_KEY"])
    
    See README.md for full reproduction instructions.
    """
    def __init__(self):
        # Set via environment variable: export GATEWAY_AUTH_URL="your_url"
        self.url     = os.environ.get("GATEWAY_AUTH_URL", "YOUR_GATEWAY_AUTH_URL")
        self.payload = os.environ.get("GATEWAY_CREDENTIALS", "YOUR_CREDENTIALS")
        self.headers = {'Content-Type': 'text/plain'}

    def get_token(self):
        for attempt in range(4):
            try:
                response = requests.post(self.url, headers=self.headers,
                                         data=self.payload, timeout=120)
                if response.status_code == 200:
                    data = response.json()
                    return f"Bearer {data['access_token']}", 200
                print(f"Token API failed: {response.status_code}")
            except Exception as e:
                print("Token Fetch Error:", str(e))
            time.sleep(1)
        return None, 408


class AsyncLLM:
    def __init__(self, environment="QA"):
        # Set via environment variable: export GATEWAY_URL="your_url"
        self.url = os.environ.get(
            "GATEWAY_URL",
            "YOUR_GATEWAY_URL/v1/chat/completion"
        )
        self.token   = None
        self.headers = {'Content-Type': 'application/json', 'Authorization': ''}

    def fetch_token(self):
        ts = TokenService()
        token, status = ts.get_token()
        if status == 200:
            self.token = token
            self.headers['Authorization'] = token
        else:
            raise Exception("Failed to fetch token — check GATEWAY_AUTH_URL and GATEWAY_CREDENTIALS env vars")

    def get_provider(self, engine):
        if "gemini" in engine.lower() or "imagen" in engine.lower():
            return "GoogleBard"
        elif "claude" in engine.lower():
            return "Anthropic"
        return None

    def build_payload(self, engine, messages, temperature, max_tokens, custom_identifier):
        payload = {
            "ModelToUse": engine,
            "AdditionalRequestParameters": {
                "Temperature": temperature,
                "MaxTokens": max_tokens,
                "TopProbabilities": 0.5,
                "PresencePenalty": 0,
                "FrequencyPenalty": 0
            },
            "CustomIdentifier": custom_identifier or "rups-experiment",
            "Messages": [
                {"Role": m.get("role", m.get("Role", "user")).capitalize(),
                 "Content": m.get("content", m.get("Content", ""))}
                for m in messages
            ]
        }
        if engine.startswith("gpt-5"):
            payload["AdditionalRequestParameters"].pop("StopSequences", None)
        provider = self.get_provider(engine)
        if provider:
            payload["LLMProviderType"] = provider
        return payload

    async def infer(self, session, engine, messages, temperature, max_tokens,
                    custom_identifier, semaphore):
        payload = self.build_payload(engine, messages, temperature, max_tokens, custom_identifier)
        async with semaphore:
            for attempt in range(3):
                try:
                    async with session.post(
                        self.url, headers=self.headers, json=payload,
                        timeout=aiohttp.ClientTimeout(total=240)
                    ) as resp:
                        if resp.status == 200:
                            return await resp.json(), 200
                        elif resp.status == 401:
                            self.fetch_token()
                            continue
                        else:
                            text = await resp.text()
                            return {"error": text}, resp.status
                except asyncio.TimeoutError:
                    print(f"  ⏱️ Timeout (attempt {attempt+1}/3)")
                    await asyncio.sleep(2)
                except Exception as e:
                    print(f"  ⚠️ Error (attempt {attempt+1}/3): {e}")
                    await asyncio.sleep(2)
        return {"error": "Max retries exceeded"}, 500

    def extract_text(self, response):
        try:
            return response["Messages"][0]["Message"]["Content"]
        except (KeyError, IndexError, TypeError):
            return None


print('✅ LLM Gateway class loaded')
print('   Note: Set GATEWAY_AUTH_URL and GATEWAY_CREDENTIALS environment variables')
print('   See README.md for alternative provider setup (OpenAI/Anthropic/Groq)')


## 📐 Cell 3 — Generation config & batch plan

In [ ]:
# GENERATOR_MODEL  = 'claude-sonnet-4.6'
# VALIDATOR_MODEL  = 'gpt-4o'
# GEN_TEMPERATURE  = 0.9
# VAL_TEMPERATURE  = 0.0
# GEN_TOKENS       = 900   # FIXED: was 4000 — gateway truncated responses mid-JSON
# VAL_TOKENS       = 300
# CONCURRENCY      = 5
# PROBLEMS_PER_CALL = 1    # FIXED: was 5 — 1 problem per call avoids truncation

# # ── Generation plan: 300 individual tasks ─────────────────────
# # Financial:        F1×34, F3×33, F5×33  = 100
# # Clinical:         F1×25, F2×25, F3×25, F4×25 = 100
# # HR/Supply chain:  F2×34, F4×33, F5×33  = 100

# PLAN = [
#     ('financial',       'F1', 'Schema blindness',    34),
#     ('financial',       'F3', 'Unit conflation',     33),
#     ('financial',       'F5', 'Spurious shortcut',   33),
#     ('clinical',        'F1', 'Schema blindness',    25),
#     ('clinical',        'F2', 'Temporal flattening', 25),
#     ('clinical',        'F3', 'Unit conflation',     25),
#     ('clinical',        'F4', 'Negation dropout',    25),
#     ('hr_supply_chain', 'F2', 'Temporal flattening', 34),
#     ('hr_supply_chain', 'F4', 'Negation dropout',    33),
#     ('hr_supply_chain', 'F5', 'Spurious shortcut',   33),
# ]

# # Expand into individual tasks — one task = one API call = one problem
# TASKS = []
# for domain, fm, fn, count in PLAN:
#     for i in range(count):
#         TASKS.append({
#             'domain': domain, 'failure_mode': fm,
#             'failure_name': fn, 'index': i + 1
#         })

# # Keep BATCHES for backward compat with other cells
# BATCHES = TASKS

# total_planned = len(TASKS)
# from collections import Counter
# print(f'✅ Generation plan: {total_planned} individual tasks (1 problem per API call)')
# print(f'   By domain      : {dict(Counter(t["domain"] for t in TASKS))}')
# print(f'   By failure mode: {dict(Counter(t["failure_mode"] for t in TASKS))}')


## 📝 Cell 4 — Generator & validator prompts

In [ ]:
# ── Domain instructions ──────────────────────────────────────
DOMAIN_INSTRUCTIONS = {
    'financial': (
        'Use realistic financial data: loan applications, credit bureau records, '
        'bond portfolios, income statements, or risk assessments. '
        'Include authentic field names from real datasets (Lending Club, UCI Credit). '
        'Values should be realistic and internally consistent.'
    ),
    'clinical': (
        'Use realistic clinical/medical data: ICU vitals, lab panels, medication records, '
        'clinical notes, or sepsis screening tools. '
        'Use authentic medical abbreviations (WBC, Hgb, Cr, BUN, INR, SpO2, GCS). '
        'Values should be clinically realistic — reference ranges must be accurate.'
    ),
    'hr_supply_chain': (
        'Use realistic HR or supply chain data: employee performance records, '
        'attrition risk profiles, procurement logs, or inventory events. '
        'Use field names from real HR datasets (IBM HR Attrition). '
        'Values should be realistic and internally consistent.'
    )
}

# ── Failure mode instructions ─────────────────────────────────
FAILURE_INSTRUCTIONS = {
    'F1': (
        'FAILURE MODE F1 — Schema blindness: context must contain abbreviated/technical '
        'field names (revol_util, bc_util, num_tl_90g_dpd_24m, WBC, INR, Cr, no_stock_option). '
        'Failure fires when model misinterprets or ignores field names, inferring meaning '
        'from values alone.'
    ),
    'F2': (
        'FAILURE MODE F2 — Temporal flattening: context must be time-ordered records '
        'where TREND direction is critical. '
        'Failure fires when model averages values or ignores the sequence direction.'
    ),
    'F3': (
        'FAILURE MODE F3 — Unit conflation: context must mix units '
        '(mg/dL vs mmol/L, bps vs %, monthly vs annual, decimal vs percentage rate). '
        'Failure fires when model compares values without converting to common units.'
    ),
    'F4': (
        'FAILURE MODE F4 — Negation dropout: context must contain significant negations '
        '(denies X, no_X=1 meaning X is absent, field=NO but context reverses meaning). '
        'Failure fires when model ignores or underweights negations.'
    ),
    'F5': (
        'FAILURE MODE F5 — Spurious shortcut: context must contain a salient category label '
        'correlated with outcome (job_role, loan_purpose, home_ownership, age, admission_type) '
        'but individual metrics contradict the label. '
        'Failure fires when model overweights the label over actual metrics.'
    )
}

# ── Variety hints — rotated to keep problems diverse ─────────
import itertools
VARIETY_POOL = {
    ('financial',       'F1'): itertools.cycle(['mortgage with CLTV/LTV/DSCR fields', 'corporate bond with YTM/OAS/Z-spread', 'credit card with util_rate/cash_adv_freq', 'SME loan with EBITDA/ICR/DSC', 'factoring invoice with AR_turnover/DSO/DPO', 'equity position with P/E, EV/EBITDA, ROE']),
    ('financial',       'F3'): itertools.cycle(['APR vs monthly vs daily rate', 'EUR and USD mixed without conversion', 'yield in bps vs percentage', 'gross vs net income in affordability check', 'quarterly vs annual revenue in ratios']),
    ('financial',       'F5'): itertools.cycle(['self-employed applicant with strong metrics', 'debt_consolidation purpose with good profile', 'applicant age 67 with excellent credit', 'restaurant industry loan with strong financials', 'high-default state applicant with low DTI']),
    ('clinical',        'F1'): itertools.cycle(['arterial blood gas: PaO2/PaCO2/HCO3/BE', 'coagulation panel: PT/aPTT/fibrinogen/D-dimer', 'liver function: ALT/AST/ALP/GGT', 'cardiac enzymes: troponin-I/CK-MB/BNP', 'thyroid panel: TSH/fT4/fT3', 'renal panel: Cr/BUN/eGFR/cystatin-C']),
    ('clinical',        'F2'): itertools.cycle(['antibiotic levels rising over 5 days with renal decline', 'post-surgical vitals showing slow bleeding over 6 hours', 'glucose across 3 days showing dawn phenomenon', 'chemotherapy toxicity worsening over cycles', 'neonatal bilirubin crossing phototherapy threshold']),
    ('clinical',        'F3'): itertools.cycle(['mmol/L and mg/dL mixed in same electrolyte panel', 'mcg/kg/min and mg/hr for vasopressor dosing', 'Celsius and Fahrenheit in temperature monitoring', 'mL/hr and drops/min for IV fluid rates']),
    ('clinical',        'F4'): itertools.cycle(['pre-op checklist with no_allergy=0 meaning allergy present', 'sepsis screen where normal WBC masks left shift', 'chest X-ray with multiple normal findings but critical absence', 'ED note with denied symptoms that are still clinically relevant']),
    ('hr_supply_chain', 'F2'): itertools.cycle(['quarterly sales declining over 6 quarters', 'monthly absenteeism rising steadily over a year', 'supply chain lead times growing each month', 'training completion rates dropping across annual reviews']),
    ('hr_supply_chain', 'F4'): itertools.cycle(['benefits checklist where no_dental=1 means dental absent', 'remote work policy where no_wfh_allowance=0 means allowance given', 'performance review where no_PIP_history=0 means PIP was issued']),
    ('hr_supply_chain', 'F5'): itertools.cycle(['Legal dept employee — low attrition label but disengaged', 'employee age 55 — retirement bias but highly satisfied', 'HQ city employee — low attrition label but underpaid', 'MBA employee — overqualified bias but highly engaged']),
}

# ── FIXED generator prompt — single problem, JSON object not array ──
GENERATOR_PROMPT = """You are building RUPS-300, an AI research benchmark on LLM reasoning failures.

Generate exactly 1 structured reasoning problem with these specifications:

DOMAIN: {domain}
{domain_instruction}

{failure_instruction}

VARIETY HINT (make the scenario about this): {variety_hint}

Output ONLY a single valid JSON object with these exact fields:
{{
  "id": "PLACEHOLDER",
  "domain": "{domain}",
  "failure_mode": "{failure_mode}",
  "failure_name": "{failure_name}",
  "structural_trigger": "<one sentence — the specific structural feature that triggers the failure>",
  "context": "<structured data as a pipe-separated table or JSON record>",
  "question": "<specific question requiring careful reading of the context>",
  "gold_answer": "<correct answer with 3-4 sentences of explicit reasoning>",
  "gold_failure_label": "<one sentence: exactly how this failure mode fires on this specific problem>",
  "adversarial_version": "<same scenario in plain English with structural trigger removed, or null>"
}}

Rules:
- context must be a table with | separators or a JSON object — not plain prose
- question must be unanswerable without reading the context
- gold_answer must explicitly explain why the failure mode is relevant here
- Respond with ONLY the JSON object. Start with {{ and end with }}"""

# ── Validator prompt ──────────────────────────────────────────
VALIDATOR_PROMPT = """Quality-check this AI benchmark problem. Respond ONLY with JSON.

PROBLEM:
{problem_json}

CRITERIA:
1. has_all_fields: id, domain, failure_mode, failure_name, structural_trigger, context, question, gold_answer, gold_failure_label all present
2. context_is_structured: context is a table or JSON record, not plain prose
3. question_requires_context: question needs the context to answer
4. gold_answer_is_detailed: gold_answer has at least 3 sentences
5. trigger_matches_mode: structural_trigger matches the failure_mode (F1=schema, F2=time, F3=units, F4=negations, F5=spurious label)

{{"pass": <true/false>, "failed_criteria": [<list>], "reason": "<one sentence or null>"}}"""

print('✅ Prompts and variety hints ready')
print(f'   Generator: {GENERATOR_MODEL} (temp={GEN_TEMPERATURE}) — 1 problem per call')
print(f'   Validator: {VALIDATOR_MODEL} (temp={VAL_TEMPERATURE})')


## 🚀 Cell 5 — Generate all 300 problems
> ~20–25 minutes. Progress shown per batch.
> Problems saved incrementally — safe to interrupt and resume.

In [ ]:
# ── COMPLETE STANDALONE GENERATION — run this single cell ─────
# Redefines everything cleanly, no dependency on previous cells

import asyncio, aiohttp, json, os, shutil
from datetime import datetime
from collections import Counter

SAVE_DIR = './results'
os.makedirs(SAVE_DIR, exist_ok=True)

GEN_MODEL   = 'claude-sonnet-4.6'
GEN_TOKENS  = 1500
GEN_TEMP    = 0.9
CONCURRENCY = 5

PLAN = [
    ('financial',       'F1', 'Schema blindness',    34),
    ('financial',       'F3', 'Unit conflation',     33),
    ('financial',       'F5', 'Spurious shortcut',   33),
    ('clinical',        'F1', 'Schema blindness',    25),
    ('clinical',        'F2', 'Temporal flattening', 25),
    ('clinical',        'F3', 'Unit conflation',     25),
    ('clinical',        'F4', 'Negation dropout',    25),
    ('hr_supply_chain', 'F2', 'Temporal flattening', 34),
    ('hr_supply_chain', 'F4', 'Negation dropout',    33),
    ('hr_supply_chain', 'F5', 'Spurious shortcut',   33),
]

TASKS = []
for domain, fm, fn, count in PLAN:
    for i in range(count):
        TASKS.append({'domain': domain, 'failure_mode': fm,
                      'failure_name': fn, 'index': i+1})

DOMAIN_CTX = {
    'financial': 'Realistic financial data: loan applications with fields like int_rate, revol_util, dti, delinq_2yrs, fico_range, grade, bc_util, pub_rec. Or bond portfolios, income statements, credit risk records.',
    'clinical':  'Realistic clinical data: ICU vitals (BP, HR, SpO2, GCS), lab panels (WBC, Hgb, Plt, Na, K, Cr, BUN, Lactate, INR), medication records, clinical notes. Reference ranges must be medically accurate.',
    'hr_supply_chain': 'Realistic HR data: employee performance records, attrition profiles with fields like job_satisfaction, work_life_balance, stock_option_level, no_overtime_pay, years_since_promotion. Or supply chain event logs.'
}

FAILURE_CTX = {
    'F1': 'Schema blindness — context must use abbreviated/technical field names (revol_util, bc_util, WBC, INR, Cr, no_stock_option). Failure fires when model misinterprets field names from values alone.',
    'F2': 'Temporal flattening — context must be time-ordered records where TREND direction is critical. Failure fires when model averages or ignores sequence direction.',
    'F3': 'Unit conflation — context must mix units (mg/dL vs mmol/L, bps vs %, monthly vs annual income, decimal vs percentage rate). Failure fires when model compares without converting.',
    'F4': 'Negation dropout — context must contain significant negations (denies X, no_X=1 meaning X absent, field=NO but context reverses it). Failure fires when model ignores negations.',
    'F5': 'Spurious shortcut — context must have a salient category label (job_role, loan_purpose, home_ownership, age) where individual metrics contradict the label. Failure fires when model overweights label over metrics.'
}

import itertools
VARIETY = {
    ('financial','F1'):       itertools.cycle(['mortgage with CLTV/LTV/DSCR','corporate bond with YTM/OAS','credit card with util_rate/cash_adv_freq','SME loan with EBITDA/ICR','equity with P/E/EV/EBITDA','factoring with AR_turnover/DSO']),
    ('financial','F3'):       itertools.cycle(['APR vs monthly vs daily rate','EUR and USD mixed','yield bps vs percentage','gross vs net income','quarterly vs annual revenue']),
    ('financial','F5'):       itertools.cycle(['self-employed strong metrics','debt_consolidation good profile','age 67 excellent credit','restaurant industry strong financials','high-default state low DTI']),
    ('clinical','F1'):        itertools.cycle(['ABG: PaO2/PaCO2/HCO3/BE','coag: PT/aPTT/fibrinogen','LFT: ALT/AST/ALP/GGT','cardiac: troponin/CK-MB/BNP','thyroid: TSH/fT4/fT3','renal: Cr/BUN/eGFR']),
    ('clinical','F2'):        itertools.cycle(['antibiotic levels rising with renal decline','post-surgical slow bleeding over hours','glucose dawn phenomenon over days','chemo toxicity worsening over cycles','neonatal bilirubin crossing threshold']),
    ('clinical','F3'):        itertools.cycle(['mmol/L and mg/dL in same panel','mcg/kg/min and mg/hr for vasopressors','Celsius and Fahrenheit mixed','mL/hr and drops/min for IV fluids']),
    ('clinical','F4'):        itertools.cycle(['pre-op no_allergy=0 means allergy present','sepsis screen normal WBC masks left shift','chest Xray critical absence among normals','ED note denied symptoms still clinically relevant']),
    ('hr_supply_chain','F2'): itertools.cycle(['quarterly sales declining 6 quarters','monthly absenteeism rising over year','supply chain lead times growing monthly','training completion dropping annually']),
    ('hr_supply_chain','F4'): itertools.cycle(['no_dental=1 means dental absent','no_wfh_allowance=0 means allowance given','no_PIP_history=0 means PIP was issued']),
    ('hr_supply_chain','F5'): itertools.cycle(['Legal dept low attrition label but disengaged','age 55 retirement bias but satisfied','HQ city low attrition but underpaid','MBA overqualified bias but engaged']),
}

PROMPT = """Generate exactly 1 structured reasoning problem for the RUPS-300 AI research benchmark.

DOMAIN: {domain}
{domain_ctx}

FAILURE MODE: {fm} — {fn}
{failure_ctx}

SCENARIO HINT: {hint}

Respond with ONLY a single JSON object. Start with {{ end with }}

{{
  "id": "PLACEHOLDER",
  "domain": "{domain}",
  "failure_mode": "{fm}",
  "failure_name": "{fn}",
  "structural_trigger": "<one sentence>",
  "context": "<pipe-separated table or JSON record>",
  "question": "<specific question needing the context>",
  "gold_answer": "<correct answer, 3-4 sentences with explicit reasoning>",
  "gold_failure_label": "<one sentence: how this failure fires on this problem>",
  "adversarial_version": "<plain English version with trigger removed, or null>"
}}"""


def parse_problem(text):
    if not text:
        return None
    text = text.strip()
    if '```' in text:
        parts = text.split('```')
        text = parts[1] if len(parts) > 1 else text
        if text.startswith('json'):
            text = text[4:]
        text = text.strip()
    s = text.find('{')
    e = text.rfind('}') + 1
    if s == -1 or e == 0:
        return None
    try:
        return json.loads(text[s:e])
    except Exception:
        # Attempt truncation repair
        partial = text[s:]
        depth, in_str, esc = 0, False, False
        for ch in partial:
            if esc: esc = False; continue
            if ch == '\\' and in_str: esc = True; continue
            if ch == '"': in_str = not in_str
            if not in_str:
                if ch == '{': depth += 1
                elif ch == '}': depth -= 1
        repair = partial.rstrip()
        if in_str: repair += '"'
        repair += '}' * max(depth, 1)
        try:
            return json.loads(repair)
        except Exception:
            return None


async def gen_one(session, sem, task, idx):
    key  = (task['domain'], task['failure_mode'])
    hint = next(VARIETY.get(key, itertools.cycle(['varied realistic scenario'])))
    prompt = PROMPT.format(
        domain=task['domain'], fm=task['failure_mode'],
        fn=task['failure_name'],
        domain_ctx=DOMAIN_CTX[task['domain']],
        failure_ctx=FAILURE_CTX[task['failure_mode']],
        hint=hint
    )
    raw, status = await llm.infer(
        session, GEN_MODEL,
        [{'role': 'user', 'content': prompt}],
        temperature=GEN_TEMP, max_tokens=GEN_TOKENS,
        custom_identifier=f'rups-{task["domain"][:3]}-{task["failure_mode"]}-{idx}',
        semaphore=sem
    )
    if status != 200:
        return None
    p = parse_problem(llm.extract_text(raw))
    if p:
        p['domain']       = task['domain']
        p['failure_mode'] = task['failure_mode']
        p['failure_name'] = task['failure_name']
    return p


async def run():
    sem      = asyncio.Semaphore(CONCURRENCY)
    results  = []
    errors   = 0
    start    = datetime.now()

    async with aiohttp.ClientSession() as session:
        for batch_start in range(0, len(TASKS), 10):
            batch   = TASKS[batch_start:batch_start+10]
            coros   = [gen_one(session, sem, t, batch_start+i)
                       for i, t in enumerate(batch)]
            outputs = await asyncio.gather(*coros)
            for p in outputs:
                if p:
                    results.append(p)
                else:
                    errors += 1

            done    = batch_start + len(batch)
            pct     = int(100 * done / len(TASKS))
            elapsed = (datetime.now() - start).seconds
            bar     = '█'*(pct//5) + '░'*(20-pct//5)
            print(f'\r  [{bar}] {done}/{len(TASKS)} ({pct}%)  '
                  f'✅ {len(results)}  ❌ {errors}  ⏱ {elapsed}s',
                  end='', flush=True)

            if len(results) > 0 and len(results) % 50 == 0:
                ckpt = f'{SAVE_DIR}/ckpt_{len(results)}.json'
                with open(ckpt,'w') as f: json.dump(results, f, indent=2)
                print(f'\n  💾 {len(results)} → {ckpt}')

    print(f'\n\n✅ Done: {len(results)}/{len(TASKS)} generated, {errors} failed')
    return results


# ── RUN ───────────────────────────────────────────────────────
all_problems = asyncio.run(run())

# ── ASSIGN CLEAN IDs & SAVE ───────────────────────────────────
domain_order = {'financial':0,'clinical':1,'hr_supply_chain':2}
all_problems.sort(key=lambda p:(domain_order.get(p['domain'],9), p['failure_mode']))

PREFIX = {'financial':'FIN','clinical':'CLIN','hr_supply_chain':'HR'}
counters = {}
for p in all_problems:
    k = f"{PREFIX[p['domain']]}-{p['failure_mode']}"
    counters[k] = counters.get(k,0) + 1
    p['id'] = f"{k}-{counters[k]:03d}"

out = f'{SAVE_DIR}/rups296_benchmark.json'
with open(out,'w',encoding='utf-8') as f:
    json.dump(all_problems, f, indent=2, ensure_ascii=False)
shutil.copy(out, './rups296_benchmark.json')

print(f'\n✅ Saved {len(all_problems)} problems → rups296_benchmark.json')
print(f'\nDistribution:')
for d in ['financial','clinical','hr_supply_chain']:
    sub = [p for p in all_problems if p['domain']==d]
    fms = dict(Counter(p['failure_mode'] for p in sub))
    print(f'  {d:<22} {len(sub):3d}  {fms}')
print(f'\n🎯 Next: load rups296_benchmark.json in RUPS_Pilot_Study_v2.ipynb Cell 4')

In [ ]:
# ── TOP-UP: fill missing Clinical F2 problems ─────────────────
# Need 12 more Clinical F2 (Temporal flattening) to reach 25

TOP_UP_TASKS = [
    {'domain': 'clinical', 'failure_mode': 'F2',
     'failure_name': 'Temporal flattening', 'index': i}
    for i in range(12)
]

# Variety hints specifically for Clinical F2
import itertools
CLIN_F2_HINTS = itertools.cycle([
    'post-op wound drainage increasing over 5 days',
    'INR rising over 4 days after warfarin initiation',
    'SpO2 gradually declining over 8 hours in pneumonia patient',
    'urine output dropping hourly in post-surgical patient',
    'fever pattern over 72 hours suggesting specific infection type',
    'rising intracranial pressure markers over 6 hours',
    'progressive QTc prolongation over 3 days on antipsychotics',
    'decreasing hemoglobin daily suggesting occult bleeding',
    'worsening acidosis over 6 hours in DKA management',
    'rising D-dimer trend over 5 days post-surgery',
    'decreasing Glasgow Coma Scale over 4 hours post-trauma',
    'escalating vasopressor requirements over 12 hours in sepsis',
])

async def run_topup():
    sem     = asyncio.Semaphore(CONCURRENCY)
    results = []
    errors  = 0

    async with aiohttp.ClientSession() as session:
        coros = [gen_one(session, sem, t, 9000+i)
                 for i, t in enumerate(TOP_UP_TASKS)]
        outputs = await asyncio.gather(*coros)
        for p in outputs:
            if p:
                results.append(p)
            else:
                errors += 1

    print(f'✅ Top-up: {len(results)} generated, {errors} failed')
    return results

# Override variety for this run
original_variety = VARIETY.get(('clinical','F2'))
VARIETY[('clinical','F2')] = CLIN_F2_HINTS

topup_problems = asyncio.run(run_topup())

VARIETY[('clinical','F2')] = original_variety  # restore

# Merge with existing problems
all_problems.extend(topup_problems)

# Re-sort and re-assign IDs
domain_order = {'financial':0,'clinical':1,'hr_supply_chain':2}
all_problems.sort(key=lambda p:(domain_order.get(p['domain'],9), p['failure_mode']))

PREFIX = {'financial':'FIN','clinical':'CLIN','hr_supply_chain':'HR'}
counters = {}
for p in all_problems:
    k = f"{PREFIX[p['domain']]}-{p['failure_mode']}"
    counters[k] = counters.get(k,0) + 1
    p['id'] = f"{k}-{counters[k]:03d}"

# Save final version
import shutil
out = f'{SAVE_DIR}/rups296_benchmark.json'
with open(out,'w',encoding='utf-8') as f:
    json.dump(all_problems, f, indent=2, ensure_ascii=False)
shutil.copy(out, './rups296_benchmark.json')

print(f'\n✅ Final RUPS-300 saved: {len(all_problems)} problems')
print(f'\nFinal distribution:')
for d in ['financial','clinical','hr_supply_chain']:
    sub = [p for p in all_problems if p['domain']==d]
    fms = dict(Counter(p['failure_mode'] for p in sub))
    print(f'  {d:<22} {len(sub):3d}  {fms}')
print(f'\n🎯 rups296_benchmark.json is ready — load it in RUPS_Pilot_Study_v2.ipynb')

In [ ]:
# ── FINAL TOP-UP: fill all remaining gaps ─────────────────────
# Gaps: FIN-F3 × 1, CLIN-F2 × 7, CLIN-F4 × 2 = 10 problems

FINAL_TOPUP = (
    [{'domain': 'financial',  'failure_mode': 'F3',
      'failure_name': 'Unit conflation',     'index': 9100+i} for i in range(1)] +
    [{'domain': 'clinical',   'failure_mode': 'F2',
      'failure_name': 'Temporal flattening', 'index': 9200+i} for i in range(10)] +  # extra buffer
    [{'domain': 'clinical',   'failure_mode': 'F4',
      'failure_name': 'Negation dropout',    'index': 9300+i} for i in range(5)]     # extra buffer
)

print(f'Running {len(FINAL_TOPUP)} tasks with buffer for failures...')

async def run_final_topup():
    sem     = asyncio.Semaphore(CONCURRENCY)
    results = []
    async with aiohttp.ClientSession() as session:
        coros   = [gen_one(session, sem, t, t['index'])
                   for t in FINAL_TOPUP]
        outputs = await asyncio.gather(*coros)
        for p in outputs:
            if p:
                results.append(p)
    print(f'✅ Generated: {len(results)}/{len(FINAL_TOPUP)}')
    return results

topup2 = asyncio.run(run_final_topup())
all_problems.extend(topup2)

# ── Trim to exact targets then save ───────────────────────────
TARGET = {
    ('financial',       'F1'): 34, ('financial',       'F3'): 33, ('financial',       'F5'): 33,
    ('clinical',        'F1'): 25, ('clinical',        'F2'): 25, ('clinical',        'F3'): 25, ('clinical',        'F4'): 25,
    ('hr_supply_chain', 'F2'): 34, ('hr_supply_chain', 'F4'): 33, ('hr_supply_chain', 'F5'): 33,
}

# Sort then trim each bucket to target
domain_order = {'financial':0,'clinical':1,'hr_supply_chain':2}
all_problems.sort(key=lambda p:(domain_order.get(p['domain'],9), p['failure_mode']))

trimmed = []
bucket_counts = {}
for p in all_problems:
    key = (p['domain'], p['failure_mode'])
    bucket_counts[key] = bucket_counts.get(key, 0)
    if bucket_counts[key] < TARGET.get(key, 999):
        trimmed.append(p)
        bucket_counts[key] += 1

# Re-assign clean IDs
PREFIX = {'financial':'FIN','clinical':'CLIN','hr_supply_chain':'HR'}
counters = {}
for p in trimmed:
    k = f"{PREFIX[p['domain']]}-{p['failure_mode']}"
    counters[k] = counters.get(k,0) + 1
    p['id'] = f"{k}-{counters[k]:03d}"

# Save
import shutil
out = f'{SAVE_DIR}/rups296_benchmark.json'
with open(out,'w',encoding='utf-8') as f:
    json.dump(trimmed, f, indent=2, ensure_ascii=False)
shutil.copy(out, './rups296_benchmark.json')

print(f'\n✅ RUPS-300 FINAL: {len(trimmed)} problems saved')
print(f'\nFinal distribution:')
for d in ['financial','clinical','hr_supply_chain']:
    sub = [p for p in trimmed if p['domain']==d]
    fms = dict(Counter(p['failure_mode'] for p in sub))
    deficit = 100 - len(sub)
    status  = '✅' if deficit == 0 else f'⚠️  {deficit} short'
    print(f'  {d:<22} {len(sub):3d}  {fms}  {status}')

total_target = sum(TARGET.values())
print(f'\n  Total: {len(trimmed)}/{total_target}')
if len(trimmed) >= 280:
    print(f'  ✅ Sufficient for publication — proceed to full experiment')
    print(f'\n🎯 Open RUPS_Pilot_Study_v2.ipynb')
    print(f'   Cell 4: change rups296_benchmark.json → rups296_benchmark.json')
    print(f'   Run Cells 1-9 for the full experiment')

## 🔍 Cell 6 — Review samples & spot-check quality

In [ ]:
import random

# Show 1 random problem per failure mode
for fm in ['F1', 'F2', 'F3', 'F4', 'F5']:
    candidates = [p for p in all_problems if p['failure_mode'] == fm]
    if not candidates:
        print(f'No problems for {fm}')
        continue
    p = random.choice(candidates)
    print(f"{'='*70}")
    print(f"SAMPLE — {p['id']} | {p['failure_mode']} — {p['failure_name']}")
    print(f"Domain  : {p['domain']}")
    print(f"Trigger : {p['structural_trigger']}")
    print(f"\nCONTEXT (first 400 chars):\n{p['context'][:400]}")
    print(f"\nQUESTION:\n{p['question']}")
    print(f"\nGOLD ANSWER (first 300 chars):\n{p['gold_answer'][:300]}")
    print(f"\nFAILURE FIRES WHEN:\n{p['gold_failure_label']}")
    has_adv = bool(p.get('adversarial_version'))
    print(f"\nAdversarial version: {'✅ present' if has_adv else '❌ missing'}")
    print()

## 📊 Cell 7 — Quality stats & balance check

In [ ]:
import pandas as pd

df = pd.DataFrame(all_problems)

print('=== RUPS-300 QUALITY REPORT ===')
print(f'\nTotal problems: {len(df)}')

print('\n--- Balance check ---')
print(df.groupby(['domain', 'failure_mode']).size().unstack(fill_value=0).to_string())

print('\n--- Field completeness ---')
required = ['id','domain','failure_mode','failure_name','structural_trigger',
            'context','question','gold_answer','gold_failure_label']
for field in required:
    missing = df[field].isna().sum() if field in df.columns else len(df)
    status  = '✅' if missing == 0 else f'⚠️  {missing} missing'
    print(f'  {field:<25} {status}')

print('\n--- Adversarial pair coverage ---')
has_adv = df['adversarial_version'].notna() & (df['adversarial_version'] != 'null')
print(f'  Problems with adversarial: {has_adv.sum()}/{len(df)} ({100*has_adv.sum()//len(df)}%)')

print('\n--- Context length distribution ---')
df['ctx_len'] = df['context'].str.len()
print(f'  Min: {df["ctx_len"].min()} chars')
print(f'  Max: {df["ctx_len"].max()} chars')
print(f'  Avg: {df["ctx_len"].mean():.0f} chars')

# Flag any very short contexts (possible generation failure)
short = df[df['ctx_len'] < 100]
if len(short) > 0:
    print(f'\n  ⚠️  {len(short)} problems have very short context (<100 chars) — review these:')
    for _, row in short.iterrows():
        print(f'    {row["id"]}: {row["context"][:80]}')
else:
    print(f'\n  ✅ All contexts have sufficient length')

print('\n--- Rejected problems breakdown ---')
if rejected:
    fail_counts = Counter(c for r in rejected for c in r['failed'])
    for criterion, count in fail_counts.most_common():
        print(f'  {criterion}: {count} rejections')
else:
    print('  No rejections — all problems passed validation')

## 💾 Cell 8 — Save final RUPS-300 benchmark
> This produces `rups296_benchmark.json` — the file you load in the main experiment notebook

In [ ]:
# ── Sort by domain then failure mode for clean ordering ───────
domain_order = {'financial': 0, 'clinical': 1, 'hr_supply_chain': 2}
all_problems_sorted = sorted(
    all_problems,
    key=lambda p: (domain_order.get(p['domain'], 9), p['failure_mode'], p['id'])
)

# ── Re-assign clean sequential IDs ───────────────────────────
id_counters = {}
for p in all_problems_sorted:
    key = f"{DOMAIN_PREFIX[p['domain']]}-{p['failure_mode']}"
    id_counters[key] = id_counters.get(key, 0) + 1
    p['id'] = f"{key}-{id_counters[key]:03d}"

# ── Save ─────────────────────────────────────────────────────
output_path = f'{SAVE_DIR}/rups296_benchmark.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_problems_sorted, f, indent=2, ensure_ascii=False)

# Also save a copy at the notebook root for easy loading in v2
import shutil
shutil.copy(output_path, './rups296_benchmark.json')

print(f'✅ RUPS-300 benchmark saved:')
print(f'   {output_path}')
print(f'   ./rups296_benchmark.json  (copy at notebook root for easy loading)')
print(f'\n   Total problems: {len(all_problems_sorted)}')
print(f'\n   Final distribution:')
for d in ['financial', 'clinical', 'hr_supply_chain']:
    subset = [p for p in all_problems_sorted if p['domain'] == d]
    fms    = dict(Counter(p['failure_mode'] for p in subset))
    print(f'   {d:<20} {len(subset):3d} problems  {fms}')

print(f'\n🎯 Next step:')
print(f'   Open RUPS_Pilot_Study_v2.ipynb')
print(f'   In Cell 4, change: open("rups296_benchmark.json")')
print(f'                  to: open("rups296_benchmark.json")')
print(f'   Then run Cells 1-9 for the full experiment!')

---
## 📌 What happens next

1. ✅ `rups296_benchmark.json` is saved in your working directory
2. Open `RUPS_Pilot_Study_v2.ipynb`
3. **Change ONE line in Cell 4:**
   ```python
   # From:
   problems = json.load(open('rups296_benchmark.json'))
   # To:
   problems = json.load(open('rups296_benchmark.json'))
   ```
4. Run Cells 1–9 — this is your **full Week 5 experiment**
   - 300 problems × 5 models × 3 conditions = **4,500 API calls**
   - Auto-annotated with GPT-4o judge = **4,500 more calls**
   - Runtime: ~2–3 hours
5. Cell 9 output = your paper's Table 1, Table 2, Table 3

---
*RUPS-300 Generator v1.0*